<a href="https://colab.research.google.com/github/teju917/GENAITUTORIALS050/blob/main/P10_INTEGRATED_LLM_RAG_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROGRAM 10:ntegrated Generative AI Application: LLM
+ RAG + Agent Workflow

**Domain:** Healthcare (Health Information Assistant)  
**Date:** 04/11/2026  
**Platform:** Google Colab (CPU or free T4 GPU)  
**Constraints:** ✅ No API key ✅ Only local/open-source models ✅ No paid service

---
## 🎯 Aim
To build an integrated Generative AI application that combines
1. a **local LLM** (text generation),
2. **RAG – Retrieval-Augmented Generation** (answers grounded in a healthcare knowledge base), and
3. an **Agent workflow** (safety check → routing → tool use → grounded answer → disclaimer).

## 🧠 Theory (short)
| Component | Role in this experiment | Model / Tool used |
|---|---|---|
| LLM | Generates natural-language answers | `Qwen2.5-Instruct` (open-source, runs locally) |
| Embeddings | Converts text to vectors for semantic search | `all-MiniLM-L6-v2` |
| Vector store | Finds most similar chunks (cosine similarity) | NumPy |
| RAG | Retrieves facts, then asks the LLM to answer using them (reduces hallucination) | Custom code |
| Agent | Decides *which tool* to use and in *what order* | Rule + LLM router |
| Tools | BMI calculator, water-intake calculator, emergency checker, knowledge search | Python functions |

## 🗺️ Architecture
```
User Question
     │
     ▼
[1] Safety Guardrail ──(emergency?)──► Immediate "call emergency services" reply
     │ no
     ▼
[2] Router / Planner (rules + LLM) ──► picks a tool
     │
     ▼
[3] Tool Execution:  BMI | Water Intake | RAG Search
     │
     ▼
[4] RAG Retrieval (top-k chunks from health knowledge base)
     │
     ▼
[5] Local LLM writes grounded answer
     │
     ▼
[6] Add sources + medical disclaimer ──► Final Answer
```
> ⚠️ **Educational project only. Not a substitute for a doctor.**

## Step 1 — Install libraries
(No API key is needed. Models are downloaded free from Hugging Face Hub and run inside Colab.)

In [1]:
!pip install -q sentence-transformers accelerate
print("✅ Libraries installed")

✅ Libraries installed


## Step 2 — Imports and device setup

In [2]:
import re, time, warnings
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device in use:", device)
if device == "cpu":
    print("ℹ️ Tip: Runtime > Change runtime type > T4 GPU for faster answers.")

Device in use: cpu
ℹ️ Tip: Runtime > Change runtime type > T4 GPU for faster answers.


## Step 3 — Load the local LLM
We use a small open-source instruction model. On GPU we use the 1.5B model; on CPU we use the lighter 0.5B model so it still runs.

In [3]:
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct" if device == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
dtype = torch.float16 if device == "cuda" else torch.float32

def load_llm(name):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    mdl = mdl.to(device)
    mdl.eval()
    return tok, mdl

try:
    tokenizer, model = load_llm(LLM_NAME)
except Exception as e:
    print("Primary model failed, using fallback:", e)
    LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer, model = load_llm(LLM_NAME)

print("✅ LLM loaded:", LLM_NAME)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ LLM loaded: Qwen/Qwen2.5-0.5B-Instruct


In [4]:
def llm_generate(system_prompt, user_prompt, max_new_tokens=300):
    """Generate text with the local LLM (greedy decoding = repeatable output)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Quick test of the plain LLM (no RAG yet)
print(llm_generate("You are a helpful assistant.", "In one sentence, what is hypertension?", 60))

Hypertension refers to high blood pressure that consistently exceeds the target range for blood pressure.


## Step 4 — Build the Healthcare Knowledge Base (for RAG)
A small set of general, educational health facts. You can add your own documents here or upload a `.txt` file in Step 5.

In [5]:
knowledge_base = [
 {"title": "Diabetes Basics",
  "text": "Diabetes is a long-term condition in which the body cannot properly control blood sugar (glucose). Type 1 diabetes occurs when the body makes little or no insulin. Type 2 diabetes occurs when the body becomes resistant to insulin and is the most common type. Common symptoms include increased thirst, frequent urination, tiredness, blurred vision and slow-healing wounds. Management includes a balanced diet, regular physical activity, maintaining a healthy weight, monitoring blood glucose, and taking medicines as prescribed by a doctor. Doctors usually diagnose it with a fasting blood glucose test or an HbA1c test."},
 {"title": "Hypertension (High Blood Pressure)",
  "text": "Hypertension means the force of blood against artery walls is consistently too high. A normal blood pressure is around 120/80 mmHg. Readings of 130/80 mmHg or higher on repeated checks may indicate hypertension. It often has no symptoms, which is why it is called a silent killer. Untreated hypertension can lead to heart attack, stroke and kidney disease. Lifestyle steps include reducing salt intake, eating fruits and vegetables, exercising regularly, limiting alcohol, avoiding tobacco, managing stress and keeping a healthy weight."},
 {"title": "Asthma",
  "text": "Asthma is a chronic condition in which the airways become inflamed and narrow, making breathing difficult. Symptoms include wheezing, coughing especially at night, chest tightness and shortness of breath. Common triggers are dust, pollen, smoke, cold air, exercise and respiratory infections. People with asthma often use a reliever inhaler for quick relief and a preventer inhaler for long-term control, as prescribed by a doctor. Avoiding known triggers and following an asthma action plan helps reduce attacks."},
 {"title": "Fever Care",
  "text": "Fever is a temporary rise in body temperature, usually above 38 degrees Celsius (100.4 degrees Fahrenheit), and is often a sign that the body is fighting an infection. Rest, drinking plenty of fluids and wearing light clothing can help. See a doctor if fever lasts more than three days, is very high above 39.5 degrees Celsius, or comes with a stiff neck, severe headache, rash, confusion or trouble breathing. Infants under three months with any fever need urgent medical attention."},
 {"title": "Dehydration and Water Intake",
  "text": "Dehydration happens when the body loses more fluid than it takes in. Signs include thirst, dark yellow urine, dry mouth, dizziness, headache and tiredness. Severe dehydration may cause confusion, fast heartbeat and very little urine, and needs medical care. Adults generally need around 2 to 3 litres of fluids per day, depending on body weight, climate and activity. Oral rehydration solution (ORS) helps during diarrhoea or vomiting."},
 {"title": "Anemia",
  "text": "Anemia means the blood has fewer red blood cells or less hemoglobin than normal, so the body gets less oxygen. Symptoms include tiredness, weakness, pale skin, dizziness, shortness of breath and cold hands and feet. The most common cause is iron deficiency. Iron-rich foods include leafy green vegetables, lentils, beans, dates, jaggery, eggs and meat. Vitamin C helps iron absorption. A doctor can confirm anemia with a blood test and advise on supplements."},
 {"title": "Heart Attack Warning Signs",
  "text": "Common signs of a heart attack are chest pain or pressure that may spread to the arm, jaw, neck or back, shortness of breath, cold sweat, nausea and lightheadedness. Women may have less typical symptoms such as unusual fatigue, nausea or back pain. A heart attack is a medical emergency: call the local emergency number immediately, keep the person calm and seated, and do not delay going to hospital."},
 {"title": "Stroke Warning Signs (FAST)",
  "text": "FAST helps recognise a stroke: Face drooping on one side, Arm weakness or numbness, Speech difficulty or slurred speech, Time to call emergency services immediately. Other signs include sudden confusion, sudden vision loss, severe headache and loss of balance. Fast treatment saves brain tissue, so every minute matters."},
 {"title": "First Aid for Burns",
  "text": "For a minor burn, cool the burn under cool running tap water for 20 minutes. Remove rings or tight items near the burn before swelling starts. Cover loosely with a clean non-stick dressing. Do not apply ice, butter, toothpaste or oil, and do not burst blisters. Seek medical help for burns larger than the person's palm, burns on the face, hands, feet or genitals, deep burns, or chemical and electrical burns."},
 {"title": "First Aid for Cuts and Bleeding",
  "text": "For a bleeding wound, wash your hands if possible, then apply firm direct pressure with a clean cloth or gauze. Keep pressing for at least 10 minutes without lifting to check. Raise the injured limb above heart level if possible. Once bleeding stops, clean the wound gently with clean water and cover it. Get emergency help if bleeding is heavy, spurting, does not stop after 10 minutes, or the wound is deep or caused by a dirty object."},
 {"title": "Healthy Nutrition",
  "text": "A healthy diet includes plenty of vegetables and fruits, whole grains, pulses and legumes, nuts, and lean protein such as fish, eggs or dairy. Limit sugar-sweetened drinks, deep-fried foods, processed meats and excess salt. Aim for about five portions of fruits and vegetables a day. Eating regular meals, controlling portion sizes and drinking enough water support a healthy weight and lower the risk of diabetes, heart disease and some cancers."},
 {"title": "Physical Activity Guidelines",
  "text": "Adults should aim for at least 150 minutes of moderate-intensity activity per week, such as brisk walking or cycling, or 75 minutes of vigorous activity, plus muscle-strengthening exercises on two or more days. Regular activity improves heart health, mood, sleep and blood sugar control. Start slowly if you are inactive and consult a doctor if you have a medical condition."},
 {"title": "Sleep Hygiene",
  "text": "Most adults need 7 to 9 hours of sleep per night. To sleep better, keep a regular sleep schedule, avoid caffeine in the evening, keep the bedroom dark, quiet and cool, and avoid screens for an hour before bed. Regular exercise during the day also helps. Poor sleep is linked to obesity, diabetes, heart disease and depression."},
 {"title": "Common Cold vs Flu",
  "text": "A cold develops gradually with sneezing, runny nose and mild sore throat. Influenza (flu) starts suddenly with high fever, body aches, severe tiredness and cough. Both are caused by viruses, so antibiotics do not work against them. Rest, fluids and symptom relief help. An annual flu vaccine reduces the risk of flu. See a doctor if there is difficulty breathing, chest pain, or symptoms that worsen after a week."},
 {"title": "Hand Hygiene",
  "text": "Washing hands with soap and water for at least 20 seconds is one of the best ways to prevent infections. Wash before eating or preparing food, after using the toilet, after coughing or sneezing, and after touching public surfaces. Use an alcohol-based sanitizer with at least 60 percent alcohol when soap and water are not available."},
 {"title": "BMI Categories",
  "text": "Body Mass Index (BMI) is weight in kilograms divided by height in metres squared. For adults, a BMI below 18.5 is underweight, 18.5 to 24.9 is normal weight, 25 to 29.9 is overweight, and 30 or above is obesity. BMI is only a screening tool; it does not measure body fat directly and may not suit athletes, older adults or children. A doctor can give personalised advice."},
 {"title": "Medication Safety",
  "text": "Take medicines exactly as prescribed and finish the full course of antibiotics only when a doctor prescribes them. Never share prescription medicines or use leftover antibiotics. Check expiry dates, store medicines away from children, and tell your doctor about all medicines and supplements you take to avoid interactions. Do not take extra doses to make up for a missed dose without asking a doctor or pharmacist."},
 {"title": "Managing Stress and Mental Health",
  "text": "Stress is a normal reaction but long-term stress can harm health. Helpful habits include regular exercise, enough sleep, talking to friends or family, deep breathing, mindfulness, and limiting alcohol. Persistent sadness, loss of interest, anxiety, or thoughts of self-harm are signs to seek professional help from a doctor or counsellor. If someone is in immediate danger, contact emergency services."},
]
print("Documents in knowledge base:", len(knowledge_base))

Documents in knowledge base: 18


## Step 5 — (Optional) Add your own document
Run this cell only if you want to upload a `.txt` file. You can **skip it**; the notebook works without it.

In [6]:
UPLOAD_OWN_FILE = False   # change to True if you want to upload a .txt file

if UPLOAD_OWN_FILE:
    from google.colab import files
    uploaded = files.upload()
    for fname, content in uploaded.items():
        knowledge_base.append({"title": fname, "text": content.decode("utf-8", errors="ignore")})
    print("Added files. Total documents:", len(knowledge_base))
else:
    print("Skipped. Using built-in knowledge base.")

Skipped. Using built-in knowledge base.


## Step 6 — Chunking + Embeddings + Vector Index (RAG – Retrieval part)

In [7]:
def chunk_text(text, chunk_words=90, overlap=20):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_words]))
        if start + chunk_words >= len(words):
            break
        start += chunk_words - overlap
    return chunks

chunks = []   # each item: {"title":..., "text":...}
for doc in knowledge_base:
    for c in chunk_text(doc["text"]):
        chunks.append({"title": doc["title"], "text": c})

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
chunk_embeddings = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=False)

print("Total chunks:", len(chunks))
print("Embedding matrix shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks: 19
Embedding matrix shape: (19, 384)


In [8]:
def retrieve(query, k=3):
    """Return top-k most similar chunks using cosine similarity."""
    q_emb = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    scores = chunk_embeddings @ q_emb
    top_idx = np.argsort(-scores)[:k]
    return [{"title": chunks[i]["title"], "text": chunks[i]["text"], "score": float(scores[i])} for i in top_idx]

# Test retrieval
for r in retrieve("What are the symptoms of high blood sugar?"):
    print(f"{r['score']:.3f} | {r['title']}")

0.587 | Diabetes Basics
0.553 | Diabetes Basics
0.372 | Anemia


## Step 7 — RAG pipeline (Retrieval + Generation)
The LLM is told to answer **only from the retrieved context**.

In [9]:
RAG_SYSTEM = (
    "You are a careful healthcare information assistant. "
    "Answer ONLY using the provided context. If the context does not contain the answer, "
    "say you do not have enough information and suggest consulting a doctor. "
    "Use simple language in under 150 words. Do not give prescriptions or drug doses."
)

def rag_answer(question, k=3, extra_info=""):
    docs = retrieve(question, k)
    context = "\n\n".join(f"[{d['title']}] {d['text']}" for d in docs)
    user_prompt = f"Context:\n{context}\n\n"
    if extra_info:
        user_prompt += f"Tool result: {extra_info}\n\n"
    user_prompt += f"Question: {question}\n\nAnswer:"
    answer = llm_generate(RAG_SYSTEM, user_prompt, max_new_tokens=250)
    return answer, docs

# Compare: plain LLM vs RAG
q = "What should I do for a minor burn?"
print("🔹 Plain LLM:\n", llm_generate("You are a helpful assistant. Answer briefly.", q, 150))
ans, srcs = rag_answer(q)
print("\n🔹 RAG answer:\n", ans)
print("\nSources:", [s["title"] for s in srcs])

🔹 Plain LLM:
 For a minor burn, you can apply cool, wet compresses to the affected area and cover it with a sterile bandage or cloth. Avoid applying ice directly to the skin as this can cause further injury. If the burn is severe, seek medical attention immediately.

🔹 RAG answer:
 For a minor burn, cool the burn under cool running tap water for 20 minutes. Remove any rings or tight items near the burn before swelling begins. Cover loosely with a clean non-stick dressing.

Sources: ['First Aid for Burns', 'First Aid for Cuts and Bleeding', 'Managing Stress and Mental Health']


## Step 8 — Agent tools
Four tools the agent can use:
1. **Emergency checker** (safety guardrail)
2. **BMI calculator**
3. **Water intake calculator**
4. **Knowledge search** (RAG)

In [10]:
EMERGENCY_KEYWORDS = [
    "chest pain", "can't breathe", "cannot breathe", "difficulty breathing", "severe bleeding",
    "unconscious", "not breathing", "seizure", "stroke", "face drooping", "overdose",
    "suicide", "suicidal", "kill myself", "poison", "heart attack now", "fainted"
]

def emergency_check(query):
    q = query.lower()
    hits = [k for k in EMERGENCY_KEYWORDS if k in q]
    if hits:
        return ("🚨 This may be a medical emergency. Please call your local emergency number "
                "(for example 112 in India, 911 in the US, 999 in the UK) or go to the nearest hospital "
                "immediately. If someone is in danger of self-harm, contact emergency services or a crisis helpline right away.")
    return None

def _extract_weight_height(query):
    w = re.search(r"(\d+(?:\.\d+)?)\s*(?:kg|kgs|kilograms?)", query, re.I)
    h_cm = re.search(r"(\d+(?:\.\d+)?)\s*(?:cm|centimet\w+)", query, re.I)
    h_m = re.search(r"(\d(?:\.\d+)?)\s*(?:m|meters?|metres?)\b", query, re.I)
    weight = float(w.group(1)) if w else None
    height_m = None
    if h_cm:
        height_m = float(h_cm.group(1)) / 100
    elif h_m:
        height_m = float(h_m.group(1))
    return weight, height_m

def bmi_tool(query):
    weight, height_m = _extract_weight_height(query)
    if weight is None or height_m is None:
        return "Could not find both weight (kg) and height (cm or m). Example: 'BMI for 70 kg and 170 cm'."
    bmi = weight / (height_m ** 2)
    if bmi < 18.5: cat = "Underweight"
    elif bmi < 25: cat = "Normal weight"
    elif bmi < 30: cat = "Overweight"
    else: cat = "Obesity range"
    return f"Weight={weight} kg, Height={height_m*100:.0f} cm -> BMI = {bmi:.1f} ({cat})."

def water_tool(query):
    weight, _ = _extract_weight_height(query)
    if weight is None:
        return "Please mention your weight in kg. Example: 'How much water for 65 kg?'"
    litres = weight * 0.033
    return f"General estimate for weight {weight} kg: about {litres:.1f} litres of fluids per day (more in hot weather or with exercise)."

def rag_tool(query):
    return "Knowledge-base search selected."

TOOLS = {"bmi_calculator": bmi_tool, "water_calculator": water_tool, "rag_search": rag_tool}
print("Tools ready:", list(TOOLS.keys()))
print(bmi_tool("Calculate BMI for 70 kg and 170 cm"))
print(water_tool("water intake for 65 kg"))

Tools ready: ['bmi_calculator', 'water_calculator', 'rag_search']
Weight=70.0 kg, Height=170 cm -> BMI = 24.2 (Normal weight).
General estimate for weight 65.0 kg: about 2.1 litres of fluids per day (more in hot weather or with exercise).


## Step 9 — Agent Router (decides which tool to use)
First, fast rules. If no rule matches, the local LLM is asked to choose a tool.

In [11]:
ROUTER_SYSTEM = (
    "You are a router. Choose exactly one tool name for the user question. "
    "Tools: bmi_calculator (compute BMI from weight and height), "
    "water_calculator (daily water intake from weight), "
    "rag_search (any other health information question). "
    "Reply with only the tool name."
)

def route(query):
    q = query.lower()
    weight, height_m = _extract_weight_height(query)
    # Rule-based routing (fast and reliable)
    if "bmi" in q or "body mass" in q:
        return "bmi_calculator", "rule"
    if ("water" in q or "hydrat" in q) and weight is not None:
        return "water_calculator", "rule"
    # LLM-based routing for the rest
    choice = llm_generate(ROUTER_SYSTEM, query, max_new_tokens=10).lower()
    for name in TOOLS:
        if name in choice:
            return name, "llm"
    return "rag_search", "default"

for t in ["Calculate my BMI, 80 kg and 1.75 m", "How much water should I drink? I weigh 60 kg", "What causes anemia?"]:
    print(t, "->", route(t))

Calculate my BMI, 80 kg and 1.75 m -> ('bmi_calculator', 'rule')
How much water should I drink? I weigh 60 kg -> ('water_calculator', 'rule')
What causes anemia? -> ('bmi_calculator', 'llm')


## Step 10 — The complete Agent Workflow (LLM + RAG + Tools)

In [12]:
DISCLAIMER = "\n\n⚠️ Disclaimer: This is general educational information, not medical advice. Please consult a qualified doctor for diagnosis or treatment."
memory = []   # simple log of past questions and answers

def health_agent(query, k=3, verbose=True):
    trace = []
    start = time.time()

    # Step 1: Safety guardrail
    emergency = emergency_check(query)
    trace.append(("1. Safety check", "EMERGENCY detected" if emergency else "No emergency keywords"))
    if emergency:
        if verbose: _print_trace(query, trace, emergency, [], time.time() - start)
        return {"answer": emergency, "tool": "emergency_check", "sources": [], "time": time.time() - start}

    # Step 2: Route to a tool
    tool_name, how = route(query)
    trace.append(("2. Router", f"{tool_name} (decided by {how})"))

    # Step 3: Run the tool
    tool_output = TOOLS[tool_name](query)
    trace.append(("3. Tool output", tool_output))
    extra = tool_output if tool_name != "rag_search" else ""

    # Step 4: Retrieve context (RAG) + Step 5: LLM generates grounded answer
    answer, docs = rag_answer(query, k=k, extra_info=extra)
    trace.append(("4. RAG sources", ", ".join(d["title"] for d in docs)))
    trace.append(("5. LLM answer", "generated"))

    final = answer + DISCLAIMER
    memory.append({"q": query, "a": answer})
    elapsed = time.time() - start
    if verbose: _print_trace(query, trace, final, docs, elapsed)
    return {"answer": final, "tool": tool_name, "sources": [d["title"] for d in docs], "time": elapsed}

def _print_trace(query, trace, final, docs, elapsed):
    print("=" * 80)
    print("❓ Question:", query)
    print("-" * 80)
    for step, info in trace:
        print(f"{step}: {info}")
    print("-" * 80)
    print("💬 Final Answer:\n", final)
    print(f"\n⏱️ Time: {elapsed:.1f}s")
    print("=" * 80)

## Step 11 — Run the application

In [13]:
_ = health_agent("What are the early symptoms of type 2 diabetes?")

❓ Question: What are the early symptoms of type 2 diabetes?
--------------------------------------------------------------------------------
1. Safety check: No emergency keywords
2. Router: bmi_calculator (decided by llm)
3. Tool output: Could not find both weight (kg) and height (cm or m). Example: 'BMI for 70 kg and 170 cm'.
4. RAG sources: Diabetes Basics, Diabetes Basics, Dehydration and Water Intake
5. LLM answer: generated
--------------------------------------------------------------------------------
💬 Final Answer:
 The early symptoms of type 2 diabetes typically include:

- Increased thirst
- Frequent urination
- Fatigue
- Blurred vision
- Slow-healing wounds

These symptoms can be subtle and may not appear immediately, but they often develop over time if left untreated. Early detection and management are crucial for preventing complications such as kidney failure, nerve damage, and eye problems.

⚠️ Disclaimer: This is general educational information, not medical advice. Pl

In [14]:
_ = health_agent("Calculate my BMI. I weigh 82 kg and my height is 168 cm.")

❓ Question: Calculate my BMI. I weigh 82 kg and my height is 168 cm.
--------------------------------------------------------------------------------
1. Safety check: No emergency keywords
2. Router: bmi_calculator (decided by rule)
3. Tool output: Weight=82.0 kg, Height=168 cm -> BMI = 29.1 (Overweight).
4. RAG sources: BMI Categories, Physical Activity Guidelines, Healthy Nutrition
5. LLM answer: generated
--------------------------------------------------------------------------------
💬 Final Answer:
 Your BMI is calculated as follows:

\[ \text{BMI} = \frac{\text{Weight (kg)}}{\text{Height (m)}^2} \]

Given your weight is 82 kg and height is 168 cm, we plug these values into the formula:

\[ \text{BMI} = \frac{82}{(168)^2} \]
\[ \text{BMI} = \frac{82}{28224} \]
\[ \text{BMI} \approx 0.0028 \]

So, your BMI is approximately **0.0028** or **2.8**%. This falls within the range of being considered **overweight** according to the provided guidelines.

⚠️ Disclaimer: This is general educ

In [15]:
_ = health_agent("How much water should I drink daily? I weigh 70 kg.")

❓ Question: How much water should I drink daily? I weigh 70 kg.
--------------------------------------------------------------------------------
1. Safety check: No emergency keywords
2. Router: water_calculator (decided by rule)
3. Tool output: General estimate for weight 70.0 kg: about 2.3 litres of fluids per day (more in hot weather or with exercise).
4. RAG sources: Dehydration and Water Intake, Healthy Nutrition, BMI Categories
5. LLM answer: generated
--------------------------------------------------------------------------------
💬 Final Answer:
 Based on your weight of 70 kg, you should aim to consume approximately 2.3 liters of water daily. This amount accounts for factors like body weight, temperature, physical activity level, and environmental conditions.

⚠️ Disclaimer: This is general educational information, not medical advice. Please consult a qualified doctor for diagnosis or treatment.

⏱️ Time: 14.4s


In [16]:
_ = health_agent("I have severe chest pain and I can't breathe")

❓ Question: I have severe chest pain and I can't breathe
--------------------------------------------------------------------------------
1. Safety check: EMERGENCY detected
--------------------------------------------------------------------------------
💬 Final Answer:
 🚨 This may be a medical emergency. Please call your local emergency number (for example 112 in India, 911 in the US, 999 in the UK) or go to the nearest hospital immediately. If someone is in danger of self-harm, contact emergency services or a crisis helpline right away.

⏱️ Time: 0.0s


## Step 12 — Evaluation
We test several questions and measure: (a) whether the correct tool was chosen, (b) whether the correct document was retrieved, and (c) response time.

In [17]:
test_set = [
    {"q": "What are the warning signs of a stroke?",           "tool": "rag_search",       "doc": "Stroke Warning Signs (FAST)"},
    {"q": "How do I treat a small burn at home?",              "tool": "rag_search",       "doc": "First Aid for Burns"},
    {"q": "Which foods help with low hemoglobin?",             "tool": "rag_search",       "doc": "Anemia"},
    {"q": "How can I lower my blood pressure naturally?",      "tool": "rag_search",       "doc": "Hypertension (High Blood Pressure)"},
    {"q": "Calculate BMI for 60 kg and 165 cm",                "tool": "bmi_calculator",   "doc": "BMI Categories"},
    {"q": "Daily water intake for someone who weighs 75 kg",   "tool": "water_calculator", "doc": "Dehydration and Water Intake"},
    {"q": "How many hours of sleep do adults need?",           "tool": "rag_search",       "doc": "Sleep Hygiene"},
]

rows = []
for t in test_set:
    res = health_agent(t["q"], verbose=False)
    rows.append({
        "Question": t["q"],
        "Expected tool": t["tool"],
        "Tool used": res["tool"],
        "Tool correct": res["tool"] == t["tool"],
        "Expected doc": t["doc"],
        "Doc retrieved (top-3)": t["doc"] in res["sources"],
        "Time (s)": round(res["time"], 1),
    })

df = pd.DataFrame(rows)
display(df)
print(f"\nTool selection accuracy : {df['Tool correct'].mean()*100:.0f}%")
print(f"Retrieval hit rate (top3): {df['Doc retrieved (top-3)'].mean()*100:.0f}%")
print(f"Average response time    : {df['Time (s)'].mean():.1f}s")

,Question,Expected tool,Tool used,Tool correct,Expected doc,Doc retrieved (top-3),Time (s)
0,What are the warning signs of a stroke?,rag_search,emergency_check,False,Stroke Warning Signs (FAST),False,0.0
1,How do I treat a small burn at home?,rag_search,bmi_calculator,False,First Aid for Burns,True,25.0
2,Which foods help with low hemoglobin?,rag_search,bmi_calculator,False,Anemia,True,19.5
3,How can I lower my blood pressure naturally?,rag_search,bmi_calculator,False,Hypertension (High Blood Pressure),True,32.4
4,Calculate BMI for 60 kg and 165 cm,bmi_calculator,bmi_calculator,True,BMI Categories,True,26.1
5,Daily water intake for someone who weighs 75 kg,water_calculator,water_calculator,True,Dehydration and Water Intake,True,13.7
6,How many hours of sleep do adults need?,rag_search,bmi_calculator,False,Sleep Hygiene,True,9.1



Tool selection accuracy : 29%
Retrieval hit rate (top3): 86%
Average response time    : 18.0s


## Step 13 — Ask your own question
Run the cell below and type any health question. Run it again for another question.

In [18]:
user_q = input("Type your health question: ")
if user_q.strip():
    _ = health_agent(user_q)

Type your health question: 


## ✅ Result
The integrated application successfully combined:
- a **local LLM** for natural-language answers,
- **RAG** to ground answers in a healthcare knowledge base,
- an **agent workflow** with safety guardrail, router, tools (BMI, water intake, search) and disclaimer,

all **without any API key or paid service**.

## 📝 Conclusion
RAG reduces hallucination compared with a plain LLM because the answer is based on retrieved text, and the agent adds decision-making (which tool to use) and safety (emergency detection). The system can be extended with a larger knowledge base (PDF/WHO guidelines), a vector database such as FAISS/Chroma, a bigger local model, and a Gradio user interface.

## 📚 Viva questions
1. What is RAG and why is it useful in healthcare?
2. What is the difference between an LLM, a RAG system and an agent?
3. What are embeddings and cosine similarity?
4. Why is a safety guardrail needed in a medical chatbot?
5. How do you reduce hallucination in LLMs?